In [1]:
import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
import os

import geopandas as gpd
import pandas as pd
from geoalchemy2 import Geometry

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

POSTGRES_UTEA['DATABASE'] = 'utea_precision'

In [14]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def get_data():
    engine = obtener_engine()
    
    query_lotes = """
        SELECT 
            lote.prop_ref prop, 
            lote.lote_ref lote,
            seg.id, 
            seg.geom, 
            seg.categoria_variacion paralelismo, 
            seg.categoria_velocidad velocidad
        FROM siembra_surcado.segmentos_lineas AS seg
        INNER JOIN siembra_surcado.data_lote AS lote ON lote.id = seg.data_lote_id
    """
    try:
        # read_postgis maneja la conexión y la conversión de geometría automáticamente
        gdf = gpd.read_postgis(
            sql=query_lotes, 
            con=engine, 
            geom_col='geom'  # Especifica cuál es la columna geométrica
        )
        return gdf
        
    except Exception as e:
        print(f"Error en el proceso: {e}")
        return None

In [13]:
gdf_lote = get_data()

In [16]:
path_shp = r'C:\Documents\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - SIEMBRA\SIEMBRA CON PRESICION\2026\SHP\segmentos.shp'

In [17]:
gdf_lote.to_file(path_shp, driver="ESRI Shapefile")

C:\Users\bismarksr\AppData\Local\Temp\ipykernel_18012\1459785648.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_lote.to_file(path_shp, driver="ESRI Shapefile")
